# 08 — End-to-end H&E/CD8 workflow

This notebook connects the public RocqiPath workflows for one reproducible
H&E/CD8 project:

1. validate H&E/CD8 pairing;
2. align CD8 to H&E;
3. stage aligned files for patch discovery;
4. extract matched patches;
5. optionally normalize H&E patches;
6. count DAB-positive cells on aligned CD8; and
7. summarize output/provenance.

Every expensive stage has an independent switch. Run one stage, inspect its
QC, then enable the next.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
PAIRS_ROOT = DATA_ROOT / "pairs"
HE_REFERENCE_DIR = DATA_ROOT / "reference"
STAGED_ALIGNED_ROOT = DATA_ROOT / "aligned_for_patches"
OUTPUT_ROOT = RESULTS_ROOT

BIOMARKER = "CD8"
REFERENCE_NAME = "he"
MOVING_NAME = "cd8"
TARGET_MAGNIFICATION = 20.0

ALIGNMENT_METHOD = "orb"
REFERENCE_SOURCE_MAGNIFICATION = None
MOVING_SOURCE_MAGNIFICATION = None

# Physical objective of the exported aligned WSI. With
# aligned_wsi_level=0 this is the H&E reference level-0 objective.
# Set it when the aligned OME-TIFF does not expose objective metadata.
ALIGNED_SOURCE_MAGNIFICATION = None

RUN_PAIR_DRY_RUN = False
RUN_ALIGNMENT = False
RUN_STAGE_HANDOFF = False
RUN_PATCH_EXTRACTION = False
RUN_STAIN_TRAIN = False
RUN_STAIN_APPLY = False
RUN_CELL_COUNT_BATCH = False


## One-time imports and configurations

A combined environment can be installed with:

```bash
python -m pip install -e ".[extraction,orb,stain,cellcount,viz]"
```

Replace `orb` with `valis` when non-rigid registration is required.


In [ ]:
from rocqipath.analysis import PositiveCellCounter
from rocqipath.config import (
    AlignmentConfig,
    CellCountingConfig,
    PatchExtractionConfig,
    StainNormalizationConfig,
)
from rocqipath.extraction import run_patch_extraction
from rocqipath.registration import run_alignment
from rocqipath.stain import (
    run_stain_normalization_apply,
    run_stain_normalization_train,
)

alignment_cfg = AlignmentConfig(
    input_dir=str(PAIRS_ROOT),
    output_dir=str(OUTPUT_ROOT),
    pair_folders=[BIOMARKER],
    reference_name=REFERENCE_NAME,
    moving_name=MOVING_NAME,
    alignment_method=ALIGNMENT_METHOD,
    target_magnification=TARGET_MAGNIFICATION,
    reference_source_magnification=REFERENCE_SOURCE_MAGNIFICATION,
    moving_source_magnification=MOVING_SOURCE_MAGNIFICATION,
    qc_enabled=True,
    qc_dpi=300,
    dry_run=True,
)

patch_cfg = PatchExtractionConfig(
    he_dir=str(HE_REFERENCE_DIR),
    aligned_dir=str(STAGED_ALIGNED_ROOT),
    output_dir=str(OUTPUT_ROOT),
    biomarker_folders=[BIOMARKER],
    reference_pattern=r"^(?P<sample_id>.+?)_he\.tiff?$",
    reference_name="he",
    moving_name="cd8",
    patch_size=512,
    stride=512,
    tissue_threshold=0.50,
    target_magnification=TARGET_MAGNIFICATION,
    reference_source_magnification=REFERENCE_SOURCE_MAGNIFICATION,
    target_source_magnification=ALIGNED_SOURCE_MAGNIFICATION,
    max_workers=4,
)

stain_cfg = StainNormalizationConfig(
    n_type="macenko",
    stains=["all"],
    fit_min_tissue=0.10,
    max_train_patches=500,
    resume=True,
)

cell_cfg = CellCountingConfig(
    output_dir=str(OUTPUT_ROOT),
    target_magnification=TARGET_MAGNIFICATION,
    source_magnification=ALIGNED_SOURCE_MAGNIFICATION,
    patch_size=512,
    tissue_threshold=0.10,
    min_cell_area=50,
    max_cell_area=1000,
)


## Stage 1 — Pair discovery and alignment

Dry run first. Remember that the current dry run logs pairs but returns an
empty result list.


In [ ]:
if RUN_PAIR_DRY_RUN:
    run_alignment(alignment_cfg)
else:
    print("Pair dry run disabled.")

if RUN_ALIGNMENT:
    run_cfg = AlignmentConfig.from_dict(
        {**alignment_cfg.to_dict(), "dry_run": False}
    )
    alignment_results = run_alignment(run_cfg)
    print(f"Aligned cases: {len(alignment_results)}")
else:
    alignment_results = []
    print("Alignment disabled.")


## Stage 2 — Alignment-to-patch handoff

This non-destructive staging function reconciles the current standardized
alignment output with the historical directory contract used by patch
discovery.


In [ ]:
import os
import shutil


def stage_one_marker(
    alignment_root: Path,
    staged_root: Path,
    biomarker: str,
    reference_name: str,
) -> list[Path]:
    marker_token = biomarker.lower()
    outputs: list[Path] = []
    for case_dir in sorted(alignment_root.glob(f"*_{marker_token}")):
        if not case_dir.is_dir():
            continue
        sample_id = case_dir.name[: -(len(marker_token) + 1)]
        hits = sorted(case_dir.glob("*.ome.tif*"))
        if not hits:
            continue
        destination = (
            staged_root
            / biomarker
            / f"{sample_id}_{reference_name}"
            / f"aligned_{marker_token}.ome.tiff"
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            try:
                os.link(hits[0], destination)
            except OSError:
                shutil.copy2(hits[0], destination)
        outputs.append(destination)
    return outputs


if RUN_STAGE_HANDOFF:
    staged_outputs = stage_one_marker(
        OUTPUT_ROOT / "alignment",
        STAGED_ALIGNED_ROOT,
        BIOMARKER,
        REFERENCE_NAME,
    )
    print(f"Staged files: {len(staged_outputs)}")
    print(*staged_outputs, sep="\n")
else:
    staged_outputs = []
    print("Handoff staging disabled.")


## Stage 3 — Matched patch extraction

Inspect one case manifest and several patch pairs before training any model.


In [ ]:
if RUN_PATCH_EXTRACTION:
    patch_summary = run_patch_extraction(patch_cfg)
    print(patch_summary)
else:
    patch_summary = None
    print("Patch extraction disabled.")


## Stage 4 — Optional H&E stain normalization

For a production split, point `NORMALIZATION_INPUT` only to training H&E
patches during fitting. Reuse the resulting weights for validation/test.


In [ ]:
NORMALIZATION_INPUT = OUTPUT_ROOT / "patch_extraction"
WEIGHTS_PATH = (
    OUTPUT_ROOT
    / "stain_normalization"
    / f"{stain_cfg.n_type}_weights.npz"
)

if RUN_STAIN_TRAIN:
    WEIGHTS_PATH = run_stain_normalization_train(
        str(NORMALIZATION_INPUT),
        str(OUTPUT_ROOT),
        stain_cfg,
    )
    print(WEIGHTS_PATH)
else:
    print("Stain training disabled.")

if RUN_STAIN_APPLY:
    apply_cfg = StainNormalizationConfig.from_dict(
        {**stain_cfg.to_dict(), "weights_path": str(WEIGHTS_PATH)}
    )
    stain_summary = run_stain_normalization_apply(
        str(NORMALIZATION_INPUT),
        str(OUTPUT_ROOT),
        apply_cfg,
    )
    print(stain_summary)
else:
    stain_summary = None
    print("Stain application disabled.")


## Stage 5 — DAB-positive cell counting

Count the staged aligned CD8 OME-TIFFs. They are already at the exported
alignment export level. Set `ALIGNED_SOURCE_MAGNIFICATION` to the physical
objective of that exported level when objective metadata is unavailable.


In [ ]:
staged_marker_root = STAGED_ALIGNED_ROOT / BIOMARKER
staged_cd8_slides = (
    sorted(staged_marker_root.rglob("*.ome.tif*"))
    if staged_marker_root.is_dir()
    else []
)
print(f"Staged CD8 slides: {len(staged_cd8_slides)}")

if RUN_CELL_COUNT_BATCH:
    counter = PositiveCellCounter(cell_cfg)
    cell_results = []
    for slide_path in staged_cd8_slides:
        cell_results.append(
            counter.count_slide(str(slide_path), label=BIOMARKER)
        )
    print(f"Counted slides: {len(cell_results)}")
else:
    cell_results = []
    print("Cell counting disabled.")


## Stage 6 — Save workflow provenance and inventory

This cell is inexpensive and can be rerun at any time. It stores the exact
configs and counts output files without reading large images.


In [ ]:
import json
from datetime import datetime, timezone

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "rocqipath_source_commit": "7e7296f",
    "biomarker": BIOMARKER,
    "alignment": alignment_cfg.to_dict(),
    "patch_extraction": patch_cfg.to_dict(),
    "stain_normalization": stain_cfg.to_dict(),
    "cell_counting": cell_cfg.to_dict(),
}

workflow_dir = OUTPUT_ROOT / "workflow"
workflow_dir.mkdir(parents=True, exist_ok=True)
provenance_path = workflow_dir / "he_cd8_workflow_config.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

inventory = {}
for module_name in (
    "alignment",
    "patch_extraction",
    "stain_normalization",
    "cell_counting",
    "visualization",
):
    module_dir = OUTPUT_ROOT / module_name
    inventory[module_name] = (
        sum(1 for path in module_dir.rglob("*") if path.is_file())
        if module_dir.is_dir()
        else 0
    )

print(json.dumps(inventory, indent=2))
print(f"Provenance: {provenance_path}")


## Final scientific QC gates

Do not advance only because a pipeline completed:

1. **Alignment** — review multiple regions and tissue boundaries.
2. **Patches** — confirm paired morphology and check edge/background rate.
3. **Normalization** — verify preserved nuclei and DAB signal.
4. **Cell counting** — validate masks and components against manual review.
5. **Provenance** — retain configs, package commit, source cohort, and
   exclusions with the analysis.
